In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import re
import json
import glob
import unicodedata


# ============================================================
# 2. INPUT / OUTPUT DIRECTORIES
# ============================================================

# Kaggle automatically mounts your dataset here.
# Change this path if your Kaggle dataset has a different name.

INPUT_DIR = "/kaggle/input/datasets/rajanghimire/nepali-raw-text-data-batch1"

OUTPUT_DIR = "/kaggle/working/nepali_cleaned_json"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 3. FIND ALL TXT FILES
# ============================================================

txt_files = sorted(
    glob.glob(
        os.path.join(INPUT_DIR, "*.txt")
    )
)

print("Total TXT files found:", len(txt_files))

for file in txt_files:
    print(os.path.basename(file))


# ============================================================
# 4. REMOVE HTML
# ============================================================

def remove_html(text):

    # Remove HTML tags
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    # Remove HTML entities such as &nbsp;
    text = re.sub(
        r"&[a-zA-Z0-9#]+;",
        " ",
        text
    )

    return text


# ============================================================
# 5. REMOVE URLS
# ============================================================

def remove_urls(text):

    # http / https / www URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


# ============================================================
# 6. CHECK WHETHER CHARACTER IS DEVANAGARI
# ============================================================

def is_devanagari(char):

    code = ord(char)

    # Devanagari Unicode block
    if 0x0900 <= code <= 0x097F:
        return True

    return False


# ============================================================
# 7. KEEP DEVANAGARI + PUNCTUATION + WHITESPACE
# ============================================================

def remove_non_devanagari(text):

    cleaned = []

    for char in text:

        # Keep Devanagari
        if is_devanagari(char):
            cleaned.append(char)

        # Keep whitespace
        elif char.isspace():
            cleaned.append(" ")

        # Keep punctuation
        elif unicodedata.category(char).startswith("P"):
            cleaned.append(char)

        # Remove everything else
        else:
            cleaned.append(" ")

    return "".join(cleaned)


# ============================================================
# 8. ADD SPACES AROUND PUNCTUATION
# ============================================================

def space_punctuation(text):

    # --------------------------------------------------------
    # Nepali danda
    # --------------------------------------------------------

    text = text.replace(
        "।",
        " । "
    )

    # --------------------------------------------------------
    # Double danda
    # --------------------------------------------------------

    text = text.replace(
        "॥",
        " ॥ "
    )

    # --------------------------------------------------------
    # Other common punctuation
    # --------------------------------------------------------

    punctuation = r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""

    for p in punctuation:

        text = text.replace(
            p,
            f" {p} "
        )

    # --------------------------------------------------------
    # Other Unicode punctuation
    # --------------------------------------------------------

    result = []

    for char in text:

        if (
            unicodedata.category(char).startswith("P")
            and char not in ["।", "॥"]
        ):

            result.append(" ")
            result.append(char)
            result.append(" ")

        else:

            result.append(char)

    text = "".join(result)

    return text


# ============================================================
# 9. NORMALIZE WHITESPACE
# ============================================================

def normalize_whitespace(text):

    # Replace newlines/tabs/etc. with spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# 10. COMPLETE CLEANING FUNCTION
# ============================================================

def clean_text(text):

    # --------------------------------------------------------
    # Step 1: Unicode normalization
    # --------------------------------------------------------

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # --------------------------------------------------------
    # Step 2: Remove HTML
    # --------------------------------------------------------

    text = remove_html(text)

    # --------------------------------------------------------
    # Step 3: Remove URLs
    # --------------------------------------------------------

    text = remove_urls(text)

    # --------------------------------------------------------
    # Step 4: Remove non-Devanagari text
    # --------------------------------------------------------

    text = remove_non_devanagari(text)

    # --------------------------------------------------------
    # Step 5: Add spaces around punctuation
    # --------------------------------------------------------

    text = space_punctuation(text)

    # --------------------------------------------------------
    # Step 6: Normalize spaces
    # --------------------------------------------------------

    text = normalize_whitespace(text)

    return text


# ============================================================
# 11. PROCESS ALL TXT FILES
# ============================================================

total_files = 0
total_entries = 0
total_words = 0


for file_number, txt_file in enumerate(
    txt_files,
    start=1
):

    filename = os.path.basename(txt_file)

    print("\n")
    print("=" * 70)
    print(
        f"PROCESSING FILE {file_number}/{len(txt_files)}"
    )
    print("=" * 70)

    print("Input:", filename)


    # ========================================================
    # READ TEXT FILE
    # ========================================================

    try:

        with open(
            txt_file,
            "r",
            encoding="utf-8"
        ) as f:

            raw_text = f.read()

    except UnicodeDecodeError:

        # Fallback for files that aren't strictly UTF-8
        with open(
            txt_file,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as f:

            raw_text = f.read()


    # ========================================================
    # SPLIT INTO LINES
    # ========================================================
    #
    # Every non-empty line becomes one JSON entry.
    #

    lines = raw_text.splitlines()


    # ========================================================
    # OUTPUT JSON FILE
    # ========================================================

    json_filename = (
        os.path.splitext(filename)[0]
        + ".json"
    )

    json_path = os.path.join(
        OUTPUT_DIR,
        json_filename
    )


    file_entries = 0
    file_words = 0


    # ========================================================
    # WRITE JSON
    # ========================================================

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write("[\n")

        first_entry = True


        for line in lines:

            # ------------------------------------------------
            # Clean line
            # ------------------------------------------------

            cleaned = clean_text(line)


            # ------------------------------------------------
            # Skip completely empty lines
            # ------------------------------------------------

            if not cleaned:
                continue


            # ------------------------------------------------
            # Word count
            # ------------------------------------------------

            wordcount = len(
                cleaned.split()
            )


            # ------------------------------------------------
            # JSON record
            # ------------------------------------------------

            record = {
                "source": "nepali-raw-text-data-batch-1",
                "url": filename,
                "text": cleaned,
                "wordcount": wordcount
            }


            # ------------------------------------------------
            # Write JSON object
            # ------------------------------------------------

            if not first_entry:
                f.write(",\n")

            json.dump(
                record,
                f,
                ensure_ascii=False,
                indent=2
            )

            first_entry = False


            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            file_entries += 1
            file_words += wordcount


        f.write("\n]")


    # ========================================================
    # UPDATE GLOBAL STATISTICS
    # ========================================================

    total_files += 1
    total_entries += file_entries
    total_words += file_words


    # ========================================================
    # DISPLAY FILE STATISTICS
    # ========================================================

    print("\nFILE COMPLETED")
    print("-" * 60)

    print(
        "File:",
        filename
    )

    print(
        "JSON entries:",
        file_entries
    )

    print(
        "Word count:",
        file_words
    )

    print(
        "JSON saved:",
        json_path
    )


# ============================================================
# 12. FINAL STATISTICS
# ============================================================

print("\n\n")
print("=" * 70)
print("CORPUS CLEANING COMPLETE")
print("=" * 70)

print(
    "Files processed:",
    total_files
)

print(
    "Total JSON entries:",
    total_entries
)

print(
    "TOTAL WORD COUNT:",
    total_words
)

if total_entries > 0:

    print(
        "Average words per entry:",
        round(
            total_words / total_entries,
            2
        )
    )

print(
    "\nOutput directory:"
)

print(OUTPUT_DIR)


# ============================================================
# 13. DISPLAY GENERATED JSON FILES
# ============================================================

json_files = sorted(
    glob.glob(
        os.path.join(
            OUTPUT_DIR,
            "*.json"
        )
    )
)

print(
    "\nJSON files generated:",
    len(json_files)
)

for file in json_files:

    print(
        os.path.basename(file)
    )

Total TXT files found: 15
ne_dedup_1.txt
ne_dedup_10.txt
ne_dedup_11.txt
ne_dedup_12.txt
ne_dedup_13.txt
ne_dedup_14.txt
ne_dedup_15.txt
ne_dedup_2.txt
ne_dedup_3.txt
ne_dedup_4.txt
ne_dedup_5.txt
ne_dedup_6.txt
ne_dedup_7.txt
ne_dedup_8.txt
ne_dedup_9.txt


PROCESSING FILE 1/15
Input: ne_dedup_1.txt

FILE COMPLETED
------------------------------------------------------------
File: ne_dedup_1.txt
JSON entries: 99997
Word count: 5105346
JSON saved: /kaggle/working/nepali_cleaned_json/ne_dedup_1.json


PROCESSING FILE 2/15
Input: ne_dedup_10.txt

FILE COMPLETED
------------------------------------------------------------
File: ne_dedup_10.txt
JSON entries: 99994
Word count: 5111752
JSON saved: /kaggle/working/nepali_cleaned_json/ne_dedup_10.json


PROCESSING FILE 3/15
Input: ne_dedup_11.txt

FILE COMPLETED
------------------------------------------------------------
File: ne_dedup_11.txt
JSON entries: 100000
Word count: 5260881
JSON saved: /kaggle/working/nepali_cleaned_json/ne_dedup_11.

In [2]:
import os
import json
import glob

# ============================================================
# INPUT / OUTPUT
# ============================================================

INPUT_DIR = "/kaggle/working/nepali_cleaned_json"

OUTPUT_FILE = "/kaggle/working/nepali_corpus_merged.json"


# ============================================================
# FIND ALL JSON FILES
# ============================================================

json_files = sorted(
    glob.glob(
        os.path.join(INPUT_DIR, "*.json")
    )
)

print("JSON files found:", len(json_files))

for file in json_files:
    print(os.path.basename(file))


# ============================================================
# MERGE ALL JSON FILES
# ============================================================

total_entries = 0
total_words = 0

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as outfile:

    outfile.write("[\n")

    first_entry = True

    for file_number, json_file in enumerate(
        json_files,
        start=1
    ):

        print(
            f"\nProcessing {file_number}/{len(json_files)}: "
            f"{os.path.basename(json_file)}"
        )

        # Read one JSON file
        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as infile:

            data = json.load(infile)


        # Add every record individually
        for record in data:

            if not first_entry:
                outfile.write(",\n")

            json.dump(
                record,
                outfile,
                ensure_ascii=False
            )

            first_entry = False

            total_entries += 1

            total_words += record.get(
                "wordcount",
                0
            )


    outfile.write("\n]")


# ============================================================
# FINAL STATISTICS
# ============================================================

print("\n")
print("=" * 70)
print("MERGING COMPLETE")
print("=" * 70)

print(
    "Files merged:",
    len(json_files)
)

print(
    "Total entries:",
    total_entries
)

print(
    "Total word count:",
    total_words
)

print(
    "Merged JSON:",
    OUTPUT_FILE
)

print(
    "File size:",
    round(
        os.path.getsize(OUTPUT_FILE) / (1024 ** 3),
        2
    ),
    "GB"
)

JSON files found: 15
ne_dedup_1.json
ne_dedup_10.json
ne_dedup_11.json
ne_dedup_12.json
ne_dedup_13.json
ne_dedup_14.json
ne_dedup_15.json
ne_dedup_2.json
ne_dedup_3.json
ne_dedup_4.json
ne_dedup_5.json
ne_dedup_6.json
ne_dedup_7.json
ne_dedup_8.json
ne_dedup_9.json

Processing 1/15: ne_dedup_1.json

Processing 2/15: ne_dedup_10.json

Processing 3/15: ne_dedup_11.json

Processing 4/15: ne_dedup_12.json

Processing 5/15: ne_dedup_13.json

Processing 6/15: ne_dedup_14.json

Processing 7/15: ne_dedup_15.json

Processing 8/15: ne_dedup_2.json

Processing 9/15: ne_dedup_3.json

Processing 10/15: ne_dedup_4.json

Processing 11/15: ne_dedup_5.json

Processing 12/15: ne_dedup_6.json

Processing 13/15: ne_dedup_7.json

Processing 14/15: ne_dedup_8.json

Processing 15/15: ne_dedup_9.json


MERGING COMPLETE
Files merged: 15
Total entries: 1490572
Total word count: 77399035
Merged JSON: /kaggle/working/nepali_corpus_merged.json
File size: 1.29 GB
